# Mean free path in an ideal hard-sphere gas

## Prerequisites

Review the [Maxwell–Boltzmann speed-distribution laboratory](maxwell-boltzmann-speed-distribution.ipynb) and the kinetic-theory relation between pressure, temperature, and number density.

## Objectives

1. Construct a unit-aware identical-particle hard-sphere gas state.
2. Evaluate collision cross section, number density, and mean free path.
3. Verify the inverse-pressure scaling at fixed temperature and collision diameter.
4. Visualize mean free path over a bounded pressure range.

The calculation is an illustrative idealized model, not experimental validation for argon.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.thermal.statmech.kinetic_theory import (
    HardSphereIdealGasState,
)
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity

In [ ]:
argon_collision_diameter_metres = 3.4e-10
pressure_pascal = 1.0
temperature_kelvin = 300.0

nominal_state = HardSphereIdealGasState(
    pressure=ScalarQuantity(
        magnitude=pressure_pascal,
        unit=PhysicalUnit(expression="pascal"),
    ),
    temperature=ScalarQuantity(
        magnitude=temperature_kelvin,
        unit=PhysicalUnit(expression="kelvin"),
    ),
    collision_diameter=ScalarQuantity(
        magnitude=argon_collision_diameter_metres,
        unit=PhysicalUnit(expression="meter"),
    ),
)

(
    nominal_state.collision_cross_section,
    nominal_state.number_density,
    nominal_state.mean_free_path,
)

In [ ]:
minimum_pressure_pascal = 1.0e-3
maximum_pressure_pascal = 1.0e5
pressure_sample_count = 401
pressure_values_pascal = np.logspace(
    np.log10(minimum_pressure_pascal),
    np.log10(maximum_pressure_pascal),
    pressure_sample_count,
    dtype=np.float64,
)
mean_free_path_values_metres = np.array(
    [
        HardSphereIdealGasState(
            pressure=ScalarQuantity(
                magnitude=float(pressure),
                unit=PhysicalUnit(expression="pascal"),
            ),
            temperature=nominal_state.temperature,
            collision_diameter=nominal_state.collision_diameter,
        ).mean_free_path.magnitude
        for pressure in pressure_values_pascal
    ],
    dtype=np.float64,
)

inverse_pressure_invariant = pressure_values_pascal * mean_free_path_values_metres
scaling_relative_tolerance = 2.0e-15
assert np.allclose(
    inverse_pressure_invariant,
    inverse_pressure_invariant[0],
    rtol=scaling_relative_tolerance,
    atol=0.0,
)

In [ ]:
figure_size_inches = (8.0, 4.5)
figure, axis = plt.subplots(figsize=figure_size_inches)
axis.loglog(
    pressure_values_pascal,
    mean_free_path_values_metres,
    label="identical hard spheres at 300 K",
)
axis.scatter(
    [nominal_state.pressure_in_pascal.magnitude],
    [nominal_state.mean_free_path.magnitude],
    color="black",
    label="1 Pa reference state",
    zorder=3,
)
axis.set(
    xlabel="pressure (Pa)",
    ylabel="mean free path (m)",
    title="Ideal hard-sphere mean free path",
)
axis.grid(True, which="both")
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

At fixed temperature and collision diameter, the ideal-gas number density is proportional to pressure. The identical-particle hard-sphere mean free path is therefore inversely proportional to pressure. The plot spans an illustrative range; it does not establish that a real gas remains ideal or that a fixed hard-sphere diameter is valid throughout that range.

## Exercises

1. Derive $\lambda=k_B T/(\sqrt{2}\,\pi d^2P)$ from the implemented properties.
2. Repeat the pressure sweep at two temperatures and explain the vertical shift.
3. Predict how doubling the collision diameter changes the mean free path.
4. Identify physical regimes where the ideal hard-sphere assumptions become inadequate.
